In [26]:
import pandas as pd
from astroquery.gaia import Gaia
import astropy.units as u
from astropy.table import Table
from astropy.coordinates import SkyCoord, Galactocentric
import matplotlib.pyplot as plt
plt.rcParams['text.usetex'] = True
import matplotlib.image as mpimg
from galpy.orbit import Orbit
from galpy.potential import MWPotential2014, evaluatePotentials
from galpy.actionAngle import actionAngleStaeckel
import numpy as np

In [27]:
cs = pd.read_csv('../data/true_sources_basta_output_final.csv',dtype={'source_id':str}) #stars with distances from basta
unconfirmed = pd.read_csv('../data/interp_d_unconfirmed_stars.csv',dtype={'source_id':str}) #stars which aren't confirmed to be RGs (pink diamonds in galaxia plot)

both = pd.concat([cs,unconfirmed])

In [28]:
#query ra, dec, and velocities from gaia
source_ids = both['source_id'].tolist()

# Process in batches of 1000
batch_size = 1000
all_results = []

for i in range(0, len(source_ids), batch_size):
    batch = source_ids[i:i+batch_size]
    id_string = ','.join(map(str, batch))
    
    query = f"""
        SELECT source_id, ra, dec,
               pmra, pmdec, radial_velocity
        FROM gaiadr3.gaia_source 
        WHERE source_id IN ({id_string})
    """
    
    job = Gaia.launch_job(query)
    all_results.append(job.get_results().to_pandas())

results_df = pd.concat(all_results, ignore_index=True)
results_df['source_id'] = results_df['source_id'].astype(str)

cs = cs.merge(results_df,on='source_id',how='left')
unconfirmed = unconfirmed.merge(results_df,on='source_id',how='left')

In [29]:
#convert to xyz coordinates (cs)
cs_coords = SkyCoord(
    ra=cs['ra'].values * u.deg,
    dec=cs['dec'].values * u.deg,
    distance=cs['distance'].values * u.pc,
    frame='icrs'
)

gc_coords = cs_coords.transform_to(Galactocentric())

cs['X'] = gc_coords.x.to(u.kpc).value
cs['Y'] = gc_coords.y.to(u.kpc).value
cs['Z'] = gc_coords.z.to(u.kpc).value

cs.to_csv('../data/true_sources_final_XYZ.csv',index=False)

In [30]:
#convert to xyz coordinates (unconfirmed sample)
unc_coords = SkyCoord(
    ra=unconfirmed['ra'].values * u.deg,
    dec = unconfirmed['dec'].values * u.deg,
    distance = unconfirmed['dist_kpc_est'].values * u.kpc,
    frame = 'icrs'
)

gc_coords = unc_coords.transform_to(Galactocentric())

unconfirmed['X'] = gc_coords.x.to(u.kpc).value
unconfirmed['Y'] = gc_coords.y.to(u.kpc).value
unconfirmed['Z'] = gc_coords.z.to(u.kpc).value

unconfirmed.to_csv('../data/unconfirmed_XYZ.csv',index=False)